In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import itertools

# Set seed for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [2]:
# --- C1. Dataset & Augmentation: Canonical Base Patterns (26 x 25 matrix) ---
# Each row is a flattened 5x5 binary grid (0=blank, 1=ink).
# These are hand-designed canonical patterns. The augmentation will create the required variants.

# Patterns are defined row-major, flattened: [R1C1, R1C2, ..., R5C5]
CANONICAL_PATTERNS = {
    'A': [0,1,1,1,0, 1,0,0,0,1, 1,1,1,1,1, 1,0,0,0,1, 1,0,0,0,1],
    'B': [1,1,1,1,0, 1,0,0,0,1, 1,1,1,1,0, 1,0,0,0,1, 1,1,1,1,0],
    'C': [0,1,1,1,1, 1,0,0,0,0, 1,0,0,0,0, 1,0,0,0,0, 0,1,1,1,1],
    'D': [1,1,1,1,0, 1,0,0,0,1, 1,0,0,0,1, 1,0,0,0,1, 1,1,1,1,0],
    'E': [1,1,1,1,1, 1,0,0,0,0, 1,1,1,0,0, 1,0,0,0,0, 1,1,1,1,1],
    'F': [1,1,1,1,1, 1,0,0,0,0, 1,1,1,0,0, 1,0,0,0,0, 1,0,0,0,0],
    'G': [0,1,1,1,1, 1,0,0,0,0, 1,0,1,1,1, 1,0,0,0,1, 0,1,1,1,1],
    'H': [1,0,0,0,1, 1,0,0,0,1, 1,1,1,1,1, 1,0,0,0,1, 1,0,0,0,1],
    'I': [1,1,1,1,1, 0,0,1,0,0, 0,0,1,0,0, 0,0,1,0,0, 1,1,1,1,1],
    'J': [0,0,0,0,1, 0,0,0,0,1, 1,0,0,0,1, 1,0,0,0,1, 0,1,1,1,0],
    'K': [1,0,0,0,1, 1,0,0,1,0, 1,1,1,0,0, 1,0,0,1,0, 1,0,0,0,1],
    'L': [1,0,0,0,0, 1,0,0,0,0, 1,0,0,0,0, 1,0,0,0,0, 1,1,1,1,1],
    'M': [1,0,0,0,1, 1,1,0,1,1, 1,0,1,0,1, 1,0,0,0,1, 1,0,0,0,1],
    'N': [1,0,0,0,1, 1,1,0,0,1, 1,0,1,0,1, 1,0,0,1,1, 1,0,0,0,1],
    'O': [0,1,1,1,0, 1,0,0,0,1, 1,0,0,0,1, 1,0,0,0,1, 0,1,1,1,0],
    'P': [1,1,1,1,0, 1,0,0,0,1, 1,1,1,1,0, 1,0,0,0,0, 1,0,0,0,0],
    'Q': [0,1,1,1,0, 1,0,0,0,1, 1,0,1,0,1, 1,0,0,1,0, 0,1,1,1,1], # Q with tail
    'R': [1,1,1,1,0, 1,0,0,0,1, 1,1,1,1,0, 1,0,0,1,0, 1,0,0,0,1],
    'S': [0,1,1,1,1, 1,0,0,0,0, 0,1,1,1,0, 0,0,0,0,1, 1,1,1,1,0],
    'T': [1,1,1,1,1, 0,0,1,0,0, 0,0,1,0,0, 0,0,1,0,0, 0,0,1,0,0],
    'U': [1,0,0,0,1, 1,0,0,0,1, 1,0,0,0,1, 1,0,0,0,1, 0,1,1,1,0],
    'V': [1,0,0,0,1, 1,0,0,0,1, 1,0,0,0,1, 0,1,0,1,0, 0,0,1,0,0],
    'W': [1,0,0,0,1, 1,0,0,0,1, 1,0,1,0,1, 1,1,0,1,1, 0,1,1,1,0],
    'X': [1,0,0,0,1, 0,1,0,1,0, 0,0,1,0,0, 0,1,0,1,0, 1,0,0,0,1],
    'Y': [1,0,0,0,1, 0,1,0,1,0, 0,0,1,0,0, 0,0,1,0,0, 0,0,1,0,0],
    'Z': [1,1,1,1,1, 0,0,0,1,0, 0,0,1,0,0, 0,1,0,0,0, 1,1,1,1,1]
}
LETTER_KEYS = list(CANONICAL_PATTERNS.keys())
CANONICAL_ARRAY = np.array(list(CANONICAL_PATTERNS.values()), dtype=np.float32)

# Parameters
INPUT_DIM = 25
NUM_CLASSES = 26
P_FLIP = 0.02
N_VARIANTS_PER_BASE = 10 # Generate 10 augmented samples from each of the 26 base patterns
TOTAL_SAMPLES = NUM_CLASSES * N_VARIANTS_PER_BASE # 260 samples (meets >= 78 requirement)
VAL_SPLIT = 0.2

def shift_pattern(pattern_25, shift_x, shift_y):
    """Shifts a 5x5 pattern (up/down/left/right by 1 pixel) with no wrap."""
    pattern_5x5 = pattern_25.reshape(5, 5)

    # Initialize a blank 5x5 grid
    shifted_pattern = np.zeros((5, 5), dtype=np.float32)

    # Define the source and destination slice indices
    src_rows = slice(max(0, -shift_y), min(5, 5 - shift_y))
    dest_rows = slice(max(0, shift_y), min(5, 5 + shift_y))
    src_cols = slice(max(0, -shift_x), min(5, 5 - shift_x))
    dest_cols = slice(max(0, shift_x), min(5, 5 + shift_x))

    shifted_pattern[dest_rows, dest_cols] = pattern_5x5[src_rows, src_cols]

    return shifted_pattern.flatten()

def augment_data(base_patterns, n_variants, p_flip=P_FLIP, seed=SEED):
    """
    Generates augmented data by applying random flips and 1-pixel shifts.
    """
    np.random.seed(seed)
    X_list = []
    Y_list = []

    # Possible shifts: -1 (left/up), 0 (none), 1 (right/down)
    possible_shifts = [-1, 0, 1]

    for class_idx, base_pattern in enumerate(base_patterns):
        for _ in range(n_variants):
            current_pattern = np.copy(base_pattern)

            # 1. Random Flips (p=0.02)
            flip_mask = np.random.rand(INPUT_DIM) < p_flip
            current_pattern[flip_mask] = 1.0 - current_pattern[flip_mask]

            # 2. Random 1-pixel shifts (shift_x, shift_y in [-1, 0, 1])
            shift_x = np.random.choice(possible_shifts)
            shift_y = np.random.choice(possible_shifts)

            if shift_x != 0 or shift_y != 0:
                current_pattern = shift_pattern(current_pattern, shift_x, shift_y)

            X_list.append(current_pattern)
            Y_list.append(class_idx)

    X = np.array(X_list, dtype=np.float32)
    Y = tf.keras.utils.to_categorical(Y_list, num_classes=NUM_CLASSES)

    return X, Y

# Generate the dataset
X_DATA, Y_DATA_ONE_HOT = augment_data(CANONICAL_ARRAY, N_VARIANTS_PER_BASE)


In [3]:
# --- C2. Model & Training ---

def create_model(l2_lambda=1e-4, dropout_rate=0.2):
    """
    Architecture: [25 -> 64 -> 32 -> 26]. Includes L2 and Dropout.
    """
    model = Sequential([
        # Input layer (25) -> H1 (64) with L2 and Dropout
        Dense(64, activation='relu', input_shape=(INPUT_DIM,),
              kernel_regularizer=l2(l2_lambda), name='H1'),
        Dropout(dropout_rate),

        # H2 (32)
        Dense(32, activation='relu', name='H2'),

        # Output layer (26)
        Dense(NUM_CLASSES, activation='softmax', name='Output')
    ])

    # Optimizer: Adam (Loss: cross-entropy)
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Setup model and training parameters
model = create_model()
BATCH_SIZE = 32
EPOCHS = 300

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# Split data for training and validation
X_train, X_val, Y_train, Y_val = train_test_split(X_DATA, Y_DATA_ONE_HOT,
                                                    test_size=VAL_SPLIT,
                                                    random_state=SEED,
                                                    stratify=Y_DATA_ONE_HOT)

print("Starting training (300 epochs max with Early Stopping)...")
history = model.fit(X_train, Y_train,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_data=(X_val, Y_val),
                    callbacks=[early_stopping],
                    verbose=1)

print("\nTraining Complete.")
final_loss, final_accuracy = model.evaluate(X_val, Y_val, verbose=0)
print(f"Validation Accuracy after Early Stop: {final_accuracy:.4f}")

# Generate predictions and confusion matrix
Y_pred_one_hot = model.predict(X_val, verbose=0)
Y_pred_classes = np.argmax(Y_pred_one_hot, axis=1)
Y_true_classes = np.argmax(Y_val, axis=1)

conf_matrix = confusion_matrix(Y_true_classes, Y_pred_classes)
class_report = classification_report(Y_true_classes, Y_pred_classes, target_names=LETTER_KEYS, zero_division=0)


Starting training (300 epochs max with Early Stopping)...
Epoch 1/300


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 141ms/step - accuracy: 0.0395 - loss: 3.2876 - val_accuracy: 0.0769 - val_loss: 3.2606
Epoch 2/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.0700 - loss: 3.2755 - val_accuracy: 0.0385 - val_loss: 3.2419
Epoch 3/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.0675 - loss: 3.2299 - val_accuracy: 0.0385 - val_loss: 3.2284
Epoch 4/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.0772 - loss: 3.2152 - val_accuracy: 0.0385 - val_loss: 3.2163
Epoch 5/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.1554 - loss: 3.1799 - val_accuracy: 0.0769 - val_loss: 3.2049
Epoch 6/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1411 - loss: 3.1635 - val_accuracy: 0.0769 - val_loss: 3.1942
Epoch 7/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.0933 - loss: 3.1440 - val_accuracy: 0.0962 - val_loss: 3.1834
Epoch 8/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.1035 - loss: 3.1361 - val_accuracy: 0.1154 - val_loss: 3.1709
Epo

In [4]:
# --- C3. Error Analysis ---

def analyze_confusion(cm, classes, n_pairs=5):
    """
    Analyzes the confusion matrix to find the N most confused pairs.
    """
    num_classes = len(classes)
    conf_pairs = {}

    # Iterate through all off-diagonal elements (misclassifications)
    for i in range(num_classes):
        for j in range(num_classes):
            if i != j:
                # i is the true class, j is the predicted class
                count = cm[i, j]
                if count > 0:
                    pair = tuple(sorted((classes[i], classes[j])))
                    # We track the confusion in both directions, but pair is sorted
                    # to count A->B and B->A confusion together for simplicity

                    # Store as (True_i, Predicted_j): count
                    conf_pairs[(classes[i], classes[j])] = count

    # Sort by the confusion count (descending)
    sorted_conf_pairs = sorted(conf_pairs.items(), key=lambda item: item[1], reverse=True)

    # Group confusion counts for symmetric pairs (e.g., E/F and F/E)
    grouped_confusion = {}
    for (true, pred), count in sorted_conf_pairs:
        key = tuple(sorted((true, pred)))
        if key not in grouped_confusion:
            grouped_confusion[key] = {'total_count': 0, 'details': []}

        grouped_confusion[key]['total_count'] += count
        grouped_confusion[key]['details'].append(f"{true} -> {pred}: {count}")

    final_sorted_pairs = sorted(grouped_confusion.items(),
                                key=lambda item: item[1]['total_count'],
                                reverse=True)


    print(f"\n--- Top {n_pairs} Most Confused Pairs (Total Misclassifications) ---")

    for rank, (pair, data) in enumerate(final_sorted_pairs[:n_pairs]):
        print(f"{rank + 1}. {pair[0]}/{pair[1]} (Total Confusions: {data['total_count']})")

        # Propose a simple feature change for the top pair
        if rank == 0:
            print("\nProposed Simple Feature/Data Change to Reduce Confusion:")
            if pair[0] == 'E' and pair[1] == 'F':
                print(f"  * Confused Pair: E vs F. Both are highly similar (E=F + bottom bar).")
                print(f"  * Proposal: Increase the L2 weight regularization and/or Dropout rate.")
                print(f"    The network is overfitting to subtle noise rather than key features.")
                print(f"    Alternatively, ensure the base patterns for E have a very solid,")
                print(f"    unbroken bottom row, while F's bottom row is entirely zeroed,")
                print(f"    making the distinction clearer for augmentation.")
            elif pair[0] == 'O' and pair[1] == 'Q':
                 print(f"  * Confused Pair: O vs Q. Q is an O with a tail.")
                 print(f"  * Proposal: Increase the ink concentration on the 'tail' segment")
                 print(f"    of the canonical 'Q' and ensure the augmentation process is robust")
                 print(f"    to the shift/flip of this specific bottom-right segment.")
            else:
                 print(f"  * Proposal: Examine the canonical patterns for {pair[0]} and {pair[1]}.")
                 print(f"    Add more variants of these letters to the base dataset with slight")
                 print(f"    exaggerations of their distinguishing features (e.g., making the")
                 print(f"    diagonal lines in one letter slightly thicker).")

    return final_sorted_pairs


print("\n========================================================")
print("             C2. Classification Report (Validation Set)")
print("========================================================")
print(class_report)


# Run analysis (C3)
confused_pairs = analyze_confusion(conf_matrix, LETTER_KEYS, n_pairs=5)

print("\n========================================================")
print("             C3. Confusion Matrix (Full 26x26)")
print("========================================================")
# Print the confusion matrix visually
print("True (rows) vs Predicted (columns)")
print("Labels:", LETTER_KEYS)
print(conf_matrix)


             C2. Classification Report (Validation Set)
              precision    recall  f1-score   support

           A       1.00      0.50      0.67         2
           B       0.50      1.00      0.67         2
           C       0.00      0.00      0.00         2
           D       0.33      0.50      0.40         2
           E       1.00      1.00      1.00         2
           F       0.00      0.00      0.00         2
           G       1.00      0.50      0.67         2
           H       0.50      0.50      0.50         2
           I       0.25      0.50      0.33         2
           J       1.00      0.50      0.67         2
           K       0.00      0.00      0.00         2
           L       1.00      0.50      0.67         2
           M       0.00      0.00      0.00         2
           N       0.40      1.00      0.57         2
           O       0.25      0.50      0.33         2
           P       0.50      0.50      0.50         2
           Q       1.00 